In [1]:
!pip install -q --no-index --find-links=/kaggle/input/mabe-package xgboost==3.1.1

In [2]:
import datetime
import gc
import itertools
import json
import re
import sys
import time
import traceback
from collections import defaultdict
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

sys.path.append("/kaggle/usr/lib/mabe-f-beta")
from metric import score

In [3]:
# const
INPUT_DIR = Path("/kaggle/input/MABe-mouse-behavior-detection")
TRAIN_TRACKING_DIR = INPUT_DIR / "train_tracking"
TRAIN_ANNOTATION_DIR = INPUT_DIR / "train_annotation"
TEST_TRACKING_DIR = INPUT_DIR / "test_tracking"

WORKING_DIR = Path("/kaggle/working")

INDEX_COLS = [
    "video_id",
    "agent_mouse_id",
    "target_mouse_id",
    "video_frame",
]

BODY_PARTS = [
    "ear_left",
    "ear_right",
    "nose",
    "neck",
    "body_center",
    "lateral_left",
    "lateral_right",
    "hip_left",
    "hip_right",
    "tail_base",
    "tail_tip",
]

SELF_BEHAVIORS = [
    "biteobject",
    "climb",
    "dig",
    "exploreobject",
    "freeze",
    "genitalgroom",
    "huddle",
    "rear",
    "rest",
    "run",
    "selfgroom",
]

PAIR_BEHAVIORS = [
    "allogroom",
    "approach",
    "attack",
    "attemptmount",
    "avoid",
    "chase",
    "chaseattack",
    "defend",
    "disengage",
    "dominance",
    "dominancegroom",
    "dominancemount",
    "ejaculate",
    "escape",
    "flinch",
    "follow",
    "intromit",
    "mount",
    "reciprocalsniff",
    "shepherd",
    "sniff",
    "sniffbody",
    "sniffface",
    "sniffgenital",
    "submit",
    "tussle",
]

In [4]:
# read data
train_dataframe = pl.read_csv(INPUT_DIR / "train.csv")

In [5]:
# preprocess behavior labels
train_behavior_dataframe = (
    train_dataframe.filter(pl.col("behaviors_labeled").is_not_null())
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled").map_elements(eval, return_dtype=pl.List(pl.Utf8)).alias("behaviors_labeled_list"),
    )
    .explode("behaviors_labeled_list")
    .rename({"behaviors_labeled_list": "behaviors_labeled_element"})
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled_element").str.split(",").list[0].str.replace_all("'", "").alias("agent"),
        pl.col("behaviors_labeled_element").str.split(",").list[1].str.replace_all("'", "").alias("target"),
        pl.col("behaviors_labeled_element").str.split(",").list[2].str.replace_all("'", "").alias("behavior"),
    )
)

train_self_behavior_dataframe = train_behavior_dataframe.filter(pl.col("behavior").is_in(SELF_BEHAVIORS))
train_pair_behavior_dataframe = train_behavior_dataframe.filter(pl.col("behavior").is_in(PAIR_BEHAVIORS))

In [ ]:
%%writefile self_features.py

# ## Xử lý đặc trưng (cho bản thân agent - self)
# - Khoảng cách giữa các bộ phận cơ thể của agent (cm)
#   Tính khoảng cách giữa các cặp trong 11 bộ phận chính (hằng số: BODY_PARTS)
# 
# - Tốc độ ước tính của bộ phận (cm/s)
#   Tốc độ ước tính của ear_left, ear_right, tail_base trong các khoảng 500, 1000, 2000, 3000ms
# 
# - Độ giãn dài (Elongation)
#   (Khoảng cách nose-tail_base) / (Khoảng cách ear_left-ear_right)
#
# - Góc cơ thể (deg)
#   Góc của vector tạo bởi nose-body_center và body_center-tail_base

def make_self_features(
    metadata: dict,
    tracking: pl.DataFrame,
) -> pl.DataFrame:
    def body_parts_distance(body_part_1, body_part_2):
        # Khoảng cách giữa các bộ phận cơ thể của agent (cm)
        assert body_part_1 in BODY_PARTS
        assert body_part_2 in BODY_PARTS
        return (
            (pl.col(f"agent_x_{body_part_1}") - pl.col(f"agent_x_{body_part_2}")).pow(2)
            + (pl.col(f"agent_y_{body_part_1}") - pl.col(f"agent_y_{body_part_2}")).pow(2)
        ).sqrt() / metadata["pix_per_cm_approx"]

    def body_part_speed(body_part, period_ms):
        # Tốc độ ước tính của bộ phận (cm/s)
        assert body_part in BODY_PARTS
        window_frames = max(1, int(round(period_ms * metadata["frames_per_second"] / 1000.0)))
        return (
            ((pl.col(f"agent_x_{body_part}").diff()).pow(2) + (pl.col(f"agent_y_{body_part}").diff()).pow(2)).sqrt()
            / metadata["pix_per_cm_approx"]
            * metadata["frames_per_second"]
        ).rolling_mean(window_size=window_frames, center=True, min_samples=1)

    def elongation():
        # Độ giãn dài (Elongation)
        d1 = body_parts_distance("nose", "tail_base")
        d2 = body_parts_distance("ear_left", "ear_right")
        return d1 / (d2 + 1e-06)

    def body_angle():
        # Góc cơ thể (độ/deg - thực tế trả về cosine similarity)
        v1x = pl.col("agent_x_nose") - pl.col("agent_x_body_center")
        v1y = pl.col("agent_y_nose") - pl.col("agent_y_body_center")
        v2x = pl.col("agent_x_tail_base") - pl.col("agent_x_body_center")
        v2y = pl.col("agent_y_tail_base") - pl.col("agent_y_body_center")
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    n_mice = (
        (metadata["mouse1_strain"] is not None)
        + (metadata["mouse2_strain"] is not None)
        + (metadata["mouse3_strain"] is not None)
        + (metadata["mouse4_strain"] is not None)
    )
    start_frame = tracking.select(pl.col("video_frame").min()).item()
    end_frame = tracking.select(pl.col("video_frame").max()).item()

    result = []

    pivot = tracking.pivot(
        on=["bodypart"],
        index=["video_frame", "mouse_id"],
        values=["x", "y"],
    ).sort(["mouse_id", "video_frame"])
    pivot_trackings = {mouse_id: pivot.filter(pl.col("mouse_id") == mouse_id) for mouse_id in range(1, n_mice + 1)}

    for agent_mouse_id in range(1, n_mice + 1):
        result_element = pl.DataFrame(
            {
                "video_id": metadata["video_id"],
                "agent_mouse_id": agent_mouse_id,
                "target_mouse_id": -1,
                "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),
            },
            schema={
                "video_id": pl.Int32,
                "agent_mouse_id": pl.Int8,
                "target_mouse_id": pl.Int8,
                "video_frame": pl.Int32,
            },
        )

        pivot = pivot_trackings[agent_mouse_id].select(
            pl.col("video_frame"),
            pl.exclude("video_frame").name.prefix("agent_"),
        )
        columns = pivot.columns
        pivot = pivot.with_columns(
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}") for bp in BODY_PARTS if f"agent_x_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}") for bp in BODY_PARTS if f"agent_y_{bp}" not in columns],
        )

        features = pivot.with_columns(
            pl.lit(agent_mouse_id).alias("agent_mouse_id"),
            pl.lit(-1).alias("target_mouse_id"),
        ).select(
            pl.col("video_frame"),
            pl.col("agent_mouse_id"),
            pl.col("target_mouse_id"),
            *[
                body_parts_distance(body_part_1, body_part_2).alias(f"aa__{body_part_1}__{body_part_2}__distance")
                for body_part_1, body_part_2 in itertools.combinations(BODY_PARTS, 2)
            ],
            *[
                body_part_speed(body_part, period_ms).alias(f"agent__{body_part}__speed_{period_ms}ms")
                for body_part, period_ms in itertools.product(["ear_left", "ear_right", "tail_base"], [500, 1000, 2000, 3000])
            ],
            elongation().alias("agent__elongation"),
            body_angle().alias("agent__body_angle"),
        )

        result_element = result_element.join(
            features,
            on=["video_frame", "agent_mouse_id", "target_mouse_id"],
            how="left",
        )
        result.append(result_element)

    return pl.concat(result, how="vertical")

Writing self_features.py


In [ ]:
%%writefile pair_features.py

# ## Xử lý đặc trưng (cặp - pair)
# - Khoảng cách giữa các bộ phận cơ thể của agent-target (cm)
#   Khoảng cách kết hợp giữa 11 bộ phận chính (hằng số: BODY_PARTS) của agent và target
# 
# - Tốc độ ước tính của bộ phận agent, target (cm/s)
#   Tốc độ ước tính của ear_left, ear_right, tail_base trong các khoảng 500, 1000, 2000, 3000ms
# 
# - Độ giãn dài của agent, target
#   (Khoảng cách nose-tail_base) / (Khoảng cách ear_left-ear_right)
#
# - Góc cơ thể của agent, target (deg)
#   Góc của vector tạo bởi nose-body_center và body_center-tail_base

def make_pair_features(
    metadata: dict,
    tracking: pl.DataFrame,
) -> pl.DataFrame:
    def body_parts_distance(agent_or_target_1, body_part_1, agent_or_target_2, body_part_2):
        # Khoảng cách giữa các bộ phận cơ thể agent-target (cm)
        assert agent_or_target_1 == "agent" or agent_or_target_1 == "target"
        assert agent_or_target_2 == "agent" or agent_or_target_2 == "target"
        assert body_part_1 in BODY_PARTS
        assert body_part_2 in BODY_PARTS
        return (
            (pl.col(f"{agent_or_target_1}_x_{body_part_1}") - pl.col(f"{agent_or_target_2}_x_{body_part_2}")).pow(2)
            + (pl.col(f"{agent_or_target_1}_y_{body_part_1}") - pl.col(f"{agent_or_target_2}_y_{body_part_2}")).pow(2)
        ).sqrt() / metadata["pix_per_cm_approx"]

    def body_part_speed(agent_or_target, body_part, period_ms):
        # Tốc độ ước tính của bộ phận (cm/s)
        assert agent_or_target == "agent" or agent_or_target == "target"
        assert body_part in BODY_PARTS
        window_frames = max(1, int(round(period_ms * metadata["frames_per_second"] / 1000.0)))
        return (
            (
                (pl.col(f"{agent_or_target}_x_{body_part}").diff()).pow(2)
                + (pl.col(f"{agent_or_target}_y_{body_part}").diff()).pow(2)
            ).sqrt()
            / metadata["pix_per_cm_approx"]
            * metadata["frames_per_second"]
        ).rolling_mean(window_size=window_frames, center=True)

    def elongation(agent_or_target):
        # Độ giãn dài (cm)
        assert agent_or_target == "agent" or agent_or_target == "target"
        d1 = body_parts_distance(agent_or_target, "nose", agent_or_target, "tail_base")
        d2 = body_parts_distance(agent_or_target, "ear_left", agent_or_target, "ear_right")
        return d1 / (d2 + 1e-06)

    def body_angle(agent_or_target):
        # Góc cơ thể (deg)
        assert agent_or_target == "agent" or agent_or_target == "target"
        v1x = pl.col(f"{agent_or_target}_x_nose") - pl.col(f"{agent_or_target}_x_body_center")
        v1y = pl.col(f"{agent_or_target}_y_nose") - pl.col(f"{agent_or_target}_y_body_center")
        v2x = pl.col(f"{agent_or_target}_x_tail_base") - pl.col(f"{agent_or_target}_x_body_center")
        v2y = pl.col(f"{agent_or_target}_y_tail_base") - pl.col(f"{agent_or_target}_y_body_center")
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    def body_center_distance_rolling_agg(agg, period_ms):
        # Đặc trưng tổng hợp trượt (rolling aggregate) của khoảng cách
        assert agg in ["mean", "std", "var", "min", "max"] # Hàm tổng hợp
        expr = body_parts_distance("agent", "body_center", "target", "body_center")
        window_frames = max(1, int(round(period_ms * metadata["frames_per_second"] / 1000.0)))

        if agg == "mean":
            return expr.rolling_mean(window_size=window_frames, center=True, min_samples=1)
        elif agg == "std":
            return expr.rolling_std(window_size=window_frames, center=True, min_samples=1)
        elif agg == "var":
            return expr.rolling_var(window_size=window_frames, center=True, min_samples=1)
        elif agg == "min":
            return expr.rolling_min(window_size=window_frames, center=True, min_samples=1)
        elif agg == "max":
            return expr.rolling_max(window_size=window_frames, center=True, min_samples=1)
        else:
            raise ValueError()

    n_mice = (
        (metadata["mouse1_strain"] is not None)
        + (metadata["mouse2_strain"] is not None)
        + (metadata["mouse3_strain"] is not None)
        + (metadata["mouse4_strain"] is not None)
    )
    start_frame = tracking.select(pl.col("video_frame").min()).item()
    end_frame = tracking.select(pl.col("video_frame").max()).item()

    result = []

    pivot = tracking.pivot(
        on=["bodypart"],
        index=["video_frame", "mouse_id"],
        values=["x", "y"],
    ).sort(["mouse_id", "video_frame"])
    pivot_trackings = {mouse_id: pivot.filter(pl.col("mouse_id") == mouse_id) for mouse_id in range(1, n_mice + 1)}

    for agent_mouse_id, target_mouse_id in itertools.permutations(range(1, n_mice + 1), 2):
        result_element = pl.DataFrame(
            {
                "video_id": metadata["video_id"],
                "agent_mouse_id": agent_mouse_id,
                "target_mouse_id": target_mouse_id,
                "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),
            },
            schema={
                "video_id": pl.Int32,
                "agent_mouse_id": pl.Int8,
                "target_mouse_id": pl.Int8,
                "video_frame": pl.Int32,
            },
        )

        merged_pivot = (
            pivot_trackings[agent_mouse_id]
            .select(
                pl.col("video_frame"),
                pl.exclude("video_frame").name.prefix("agent_"),
            )
            .join(
                pivot_trackings[target_mouse_id].select(
                    pl.col("video_frame"),
                    pl.exclude("video_frame").name.prefix("target_"),
                ),
                on="video_frame",
                how="inner",
            )
        )
        columns = merged_pivot.columns
        merged_pivot = merged_pivot.with_columns(
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}") for bp in BODY_PARTS if f"agent_x_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}") for bp in BODY_PARTS if f"agent_y_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_x_{bp}") for bp in BODY_PARTS if f"target_x_{bp}" not in columns],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_y_{bp}") for bp in BODY_PARTS if f"target_y_{bp}" not in columns],
        )

        features = merged_pivot.with_columns(
            pl.lit(agent_mouse_id).alias("agent_mouse_id"),
            pl.lit(target_mouse_id).alias("target_mouse_id"),
        ).select(
            pl.col("video_frame"),
            pl.col("agent_mouse_id"),
            pl.col("target_mouse_id"),
            *[
                body_parts_distance("agent", agent_body_part, "target", target_body_part).alias(
                    f"at__{agent_body_part}__{target_body_part}__distance"
                )
                for agent_body_part, target_body_part in itertools.product(BODY_PARTS, repeat=2)
            ],
            *[
                body_part_speed("agent", body_part, period_ms).alias(f"agent__{body_part}__speed_{period_ms}ms")
                for body_part, period_ms in itertools.product(["ear_left", "ear_right", "tail_base"], [500, 1000, 2000, 3000])
            ],
            *[
                body_part_speed("target", body_part, period_ms).alias(f"target__{body_part}__speed_{period_ms}ms")
                for body_part, period_ms in itertools.product(["ear_left", "ear_right", "tail_base"], [500, 1000, 2000, 3000])
            ],
            elongation("agent").alias("agent__elongation"),
            elongation("target").alias("target__elongation"),
            body_angle("agent").alias("agent__body_angle"),
            body_angle("target").alias("target__body_angle"),
        )

        result_element = result_element.join(
            features,
            on=["video_frame", "agent_mouse_id", "target_mouse_id"],
            how="left",
        )
        result.append(result_element)

    return pl.concat(result, how="vertical")

Writing pair_features.py


In [8]:
%run -i self_features.py
%run -i pair_features.py

def process_video(row):
    """Process a single video to extract self and pair features."""
    lab_id = row["lab_id"]
    video_id = row["video_id"]

    tracking_path = TRAIN_TRACKING_DIR / f"{lab_id}/{video_id}.parquet"
    tracking = pl.read_parquet(tracking_path)

    self_features = make_self_features(metadata=row, tracking=tracking)
    pair_features = make_pair_features(metadata=row, tracking=tracking)

    self_features.write_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet")
    pair_features.write_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet")

    return video_id


# make data
(WORKING_DIR / "self_features").mkdir(exist_ok=True, parents=True)
(WORKING_DIR / "pair_features").mkdir(exist_ok=True, parents=True)

rows = list(train_dataframe.filter(pl.col("behaviors_labeled").is_not_null()).rows(named=True))
results = joblib.Parallel(n_jobs=-1, verbose=5)(joblib.delayed(process_video)(row) for row in rows)

print(f"Processed {len(results)} videos successfully")

del rows, results
gc.collect()

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   17.4s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  3.5min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed:  4.5min


Processed 848 videos successfully


[Parallel(n_jobs=-1)]: Done 848 out of 848 | elapsed:  5.5min finished


27

In [9]:
def tune_threshold(oof_action, y_action):
    thresholds = np.arange(0, 1.005, 0.005)
    scores = [f1_score(y_action, (oof_action >= th), zero_division=0) for th in thresholds]
    best_idx = np.argmax(scores)
    return thresholds[best_idx]

In [ ]:
def train_validate(lab_id: str, behavior: str, indices: pl.DataFrame, features: pl.DataFrame, labels: pl.Series):
    # Tạo đường dẫn thư mục để lưu kết quả
    result_dir = WORKING_DIR / "results" / lab_id / behavior
    # Tạo thư mục nếu chưa tồn tại (bao gồm cả thư mục cha)
    result_dir.mkdir(exist_ok=True, parents=True)

    # Xử lý trường hợp tổng nhãn là 0 (không có mẫu dương tính nào - Positive example)
    if labels.sum() == 0:
        # Lưu điểm F1 là 0.0
        with open(result_dir / "f1.txt", "w") as f:
            f.write("0.0\n")
        # Tạo DataFrame kết quả với tất cả giá trị dự đoán là 0
        oof_prediction_dataframe = indices.with_columns(
            pl.Series("fold", [-1] * len(labels), dtype=pl.Int8),  # Số thứ tự fold (-1 nghĩa là không sử dụng)
            pl.Series("prediction", [0.0] * len(labels), dtype=pl.Float32),  # Xác suất dự đoán
            pl.Series("predicted_label", [0] * len(labels), dtype=pl.Int8),  # Nhãn dự đoán
        )
        # Lưu kết quả dưới dạng parquet
        oof_prediction_dataframe.write_parquet(result_dir / "oof_predictions.parquet")
        return 0.0

    # Khởi tạo mảng để lưu kết quả dự đoán Out-of-Fold
    folds = np.ones(len(labels), dtype=np.int8) * -1  # Số thứ tự fold mà mỗi mẫu thuộc về
    oof_predictions = np.zeros(len(labels), dtype=np.float32)  # Xác suất dự đoán
    oof_prediction_labels = np.zeros(len(labels), dtype=np.int8)  # Nhãn dự đoán (0 hoặc 1)

    # Thực hiện kiểm chứng chéo phân tầng theo nhóm (Stratified Group Cross Validation) chia 3 phần
    # StratifiedGroupKFold giữ nguyên phân phối nhãn đồng thời đảm bảo cùng một nhóm (video_id) không bị chia tách vào nhiều fold
    for fold, (train_idx, valid_idx) in enumerate(
        StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42).split(
            X=features,  # Đặc trưng (Features)
            y=labels,  # Nhãn (Labels)
            groups=indices.get_column("video_id"),  # Tiêu chí nhóm (cùng video ID sẽ ở cùng một fold)
        )
    ):
        # Tạo thư mục để lưu kết quả cho từng fold
        result_dir_fold = result_dir / f"fold_{fold}"
        result_dir_fold.mkdir(exist_ok=True, parents=True)

        # Chia dữ liệu thành tập huấn luyện và tập kiểm định
        X_train = features[train_idx]  # Đặc trưng dùng để huấn luyện
        y_train = labels[train_idx]  # Nhãn dùng để huấn luyện
        X_valid = features[valid_idx]  # Đặc trưng dùng để kiểm định
        y_valid = labels[valid_idx]  # Nhãn dùng để kiểm định

        # ==============================================================================
        # [MỚI] BẮT ĐẦU ĐOẠN OVERSAMPLING
        # ==============================================================================
        
        # Chỉ oversample nếu có ít nhất 1 mẫu dương tính trong tập train
        num_pos = y_train.sum()
        if num_pos > 0:
            # 1. Lấy chỉ mục (indices) của các mẫu Positive (Hành vi hiếm)
            # Chuyển về numpy để lọc cho nhanh
            y_train_np = y_train.to_numpy()
            pos_indices = np.where(y_train_np == 1)[0]
            
            # 2. Định nghĩa hệ số nhân (Multiplier)
            # Ví dụ: Nếu hành vi rất hiếm (< 1%), nhân bản lên 5 lần.
            # Bạn có thể điều chỉnh logic này tùy ý.
            pos_ratio = num_pos / len(y_train)
            if pos_ratio < 0.01:
                multiplier = 5  # Rất hiếm -> Nhân 5
            elif pos_ratio < 0.05:
                multiplier = 3  # Hiếm -> Nhân 3
            elif pos_ratio < 0.1:
                multiplier = 2  # Hơi hiếm -> Nhân 2
            else:
                multiplier = 1  # Đủ nhiều -> Không nhân
            
            if multiplier > 1:
                # Lọc lấy các mẫu features và labels dương tính
                X_pos = X_train[pos_indices]
                y_pos = y_train[pos_indices]
                
                # Nhân bản
                X_pos_oversampled = pl.concat([X_pos] * (multiplier - 1))
                y_pos_oversampled = pl.concat([y_pos] * (multiplier - 1))
                
                # Gộp vào tập Train gốc
                X_train = pl.concat([X_train, X_pos_oversampled])
                y_train = pl.concat([y_train, y_pos_oversampled])
                
                # print(f"Fold {fold}: Oversampled positive class {multiplier}x. New shape: {X_train.shape}")

        # ==============================================================================
        # [MỚI] KẾT THÚC ĐOẠN OVERSAMPLING
        # ==============================================================================
        
        # Tính toán trọng số để xử lý mất cân bằng lớp (class imbalance)
        # Số lượng mẫu âm / Số lượng mẫu dương = Trọng số cho mẫu dương
        scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

        # Thiết lập các siêu tham số (hyperparameters) cho XGBoost
        params = {
            "objective": "binary:logistic",  # Bài toán phân loại nhị phân
            "eval_metric": "logloss",  # Chỉ số đánh giá: Logarithmic loss
            "device": "cpu",  # Thiết bị sử dụng
            "tree_method": "hist",  # Thuật toán nhanh dựa trên histogram
            "learning_rate": 0.05,  # Tốc độ học
            "max_depth": 6,  # Độ sâu tối đa của cây
            "min_child_weight": 5,  # Trọng số tối thiểu của node con
            "subsample": 0.8,  # Tỷ lệ mẫu sử dụng cho mỗi cây
            "colsample_bytree": 0.8,  # Tỷ lệ đặc trưng sử dụng cho mỗi cây
            "scale_pos_weight": scale_pos_weight,  # Trọng số của mẫu dương tính
            "max_bin": 64,  # Số lượng bin của histogram
            "seed": 42,  # Hạt giống ngẫu nhiên (seed)
        }
        
        # Tạo ma trận dữ liệu cho XGBoost (Tập huấn luyện dùng QuantileDMatrix, tập kiểm định dùng DMatrix thường)
        dtrain = xgb.QuantileDMatrix(X_train, label=y_train, feature_names=features.columns, max_bin=64)
        dvalid = xgb.DMatrix(X_valid, label=y_valid, feature_names=features.columns)

        # Dictionary để lưu kết quả đánh giá
        evals_result = {}
        
        # Thiết lập callback dừng sớm (early stopping)
        # Dừng huấn luyện nếu logloss trên tập kiểm định không cải thiện trong 10 vòng
        early_stopping_callback = xgb.callback.EarlyStopping(
            rounds=10,  # Số vòng liên tiếp không thấy cải thiện
            metric_name="logloss",  # Chỉ số cần theo dõi
            data_name="valid",  # Dataset cần theo dõi
            maximize=False,  # Chỉ số càng nhỏ càng tốt
            save_best=True,  # Lưu lại mô hình tốt nhất
        )
        
        # Thực hiện huấn luyện mô hình
        model = xgb.train(
            params,  # Siêu tham số
            dtrain=dtrain,  # Dữ liệu huấn luyện
            num_boost_round=250,  # Số vòng boosting tối đa
            evals=[(dtrain, "train"), (dvalid, "valid")],  # Các bộ dữ liệu để đánh giá
            callbacks=[early_stopping_callback],  # Callbacks
            evals_result=evals_result,  # Nơi lưu kết quả đánh giá
            verbose_eval=0,  # Tần suất ghi log (0 là không ghi)
        )

        # Thực hiện dự đoán trên dữ liệu kiểm định (lấy giá trị xác suất)
        fold_predictions = model.predict(dvalid)

        # Điều chỉnh ngưỡng tối ưu (threshold) để tối đa hóa điểm F1
        threshold = tune_threshold(fold_predictions, y_valid)
        
        # Lưu kết quả dự đoán Out-of-Fold
        folds[valid_idx] = fold  # Số thứ tự fold
        oof_predictions[valid_idx] = fold_predictions  # Xác suất dự đoán
        oof_prediction_labels[valid_idx] = (fold_predictions >= threshold).astype(np.int8)  # Nhị phân hóa bằng ngưỡng

        # Lưu kết quả của fold hiện tại
        # Lưu mô hình đã huấn luyện
        model.save_model(result_dir_fold / "model.json")
        # Lưu ngưỡng tối ưu
        with open(result_dir_fold / "threshold.txt", "w") as f:
            f.write(f"{threshold}\n")

        # Vẽ biểu đồ độ quan trọng của đặc trưng (Top 20, theo tiêu chí Gain)
        xgb.plot_importance(model, max_num_features=20, importance_type="gain", values_format="{v:.2f}")
        plt.tight_layout()
        plt.savefig(result_dir_fold / "feature_importance.png")
        plt.close()

        # Vẽ đường cong học tập (biến thiên của logloss)
        lgb.plot_metric(evals_result, metric="logloss")
        plt.tight_layout()
        plt.savefig(result_dir_fold / "metric.png")
        plt.close()

        # Giải phóng bộ nhớ
        gc.collect()

    # Tổng hợp kết quả dự đoán của tất cả các fold vào một DataFrame
    oof_prediction_dataframe = indices.with_columns(
        pl.Series("fold", folds, dtype=pl.Int8),  # Số thứ tự fold
        pl.Series("prediction", oof_predictions, dtype=pl.Float32),  # Xác suất dự đoán
        pl.Series("predicted_label", oof_prediction_labels, dtype=pl.Int8),  # Nhãn dự đoán
    )
    
    # Tính điểm F1 tổng thể
    f1 = f1_score(labels, oof_prediction_labels, zero_division=0)
    # Lưu điểm F1 vào file
    with open(result_dir / "f1.txt", "w") as f:
        f.write(f"{f1}\n")

    # Lưu DataFrame kết quả dự đoán
    oof_prediction_dataframe.write_parquet(result_dir / "oof_predictions.parquet")

    # Trả về điểm F1
    return f1

## Huấn luyện & Kiểm định
# Tạo mô hình (XGBoost) cho mỗi lab và hành vi (behavior)
# Huấn luyện và tính điểm F1 thông qua kiểm chứng chéo (Cross Validation 3-fold)

In [ ]:
## (Self) Huấn luyện và Kiểm định cho từng Lab-Behavior

groups = train_self_behavior_dataframe.group_by("lab_id", "behavior", maintain_order=True)
total_groups = len(list(groups))
start_time = time.perf_counter()

for idx, ((lab_id, behavior), group) in tqdm(enumerate(groups), total=total_groups):
    if idx == 0:
        tqdm.write(
            f"|{'LAB':^25}|{'BEHAVIOR':^15}|{'SAMPLES':^10}|{'POSITIVE':^10}|{'FEATURES':^10}|{'F1':^10}|{'ELAPSED TIME':^15}|",
            end="\n",
        )

    tqdm.write(f"|{lab_id:^25}|{behavior:^15}|", end="")
    index_list = []
    feature_list = []
    label_list = []

    for row in group.rows(named=True):
        video_id = row["video_id"]
        agent = row["agent"]

        agent_mouse_id = int(re.search(r"mouse(\d+)", agent).group(1))

        data = pl.scan_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet").filter(
            (pl.col("agent_mouse_id") == agent_mouse_id)
        )
        index = data.select(INDEX_COLS).collect(engine="streaming")
        feature = data.select(pl.exclude(INDEX_COLS)).collect(engine="streaming")

        # Đọc dữ liệu nhãn (annotation)
        annotation_path = TRAIN_ANNOTATION_DIR / lab_id / f"{video_id}.parquet"
        if annotation_path.exists():
            annotation = (
                pl.scan_parquet(annotation_path)
                .filter((pl.col("action") == behavior) & (pl.col("agent_id") == agent_mouse_id))
                .collect()
            )
        else:
            annotation = pl.DataFrame(
                schema={
                    "agent_id": pl.Int8,
                    "target_id": pl.Int8,
                    "action": str,
                    "start_frame": pl.Int16,
                    "stop_frame": pl.Int16,
                }
            )

        label_frames = set()
        for annotation_row in annotation.rows(named=True):
            label_frames.update(range(annotation_row["start_frame"], annotation_row["stop_frame"]))
        label = index.select(pl.col("video_frame").is_in(label_frames).cast(pl.Int8).alias("label"))

        if label.get_column("label").sum() == 0:
            continue

        index_list.append(index)
        feature_list.append(feature)
        label_list.append(label.get_column("label"))

    if not index_list:
        elapsed_time = datetime.timedelta(seconds=int(time.perf_counter() - start_time))
        tqdm.write(f"{0:>10,}|{0:>10,}|{0:>10,}|{'-':>10}|{str(elapsed_time):>15}|", end="\n")
        continue

    indices = pl.concat(index_list, how="vertical")
    features = pl.concat(feature_list, how="vertical")
    labels = pl.concat(label_list, how="vertical")

    del index_list, feature_list, label_list
    gc.collect()

    tqdm.write(f"{len(indices):>10,}|{labels.sum():>10,}|{len(features.columns):>10,}|", end="")

    f1 = train_validate(lab_id, behavior, indices, features, labels)
    tqdm.write(f"{f1:>10.2f}|", end="")

    elapsed_time = datetime.timedelta(seconds=int(time.perf_counter() - start_time))
    tqdm.write(f"{str(elapsed_time):>15}|", end="\n")

    gc.collect()

  0%|          | 0/27 [00:00<?, ?it/s]

|           LAB           |   BEHAVIOR    | SAMPLES  | POSITIVE | FEATURES |    F1    | ELAPSED TIME  |
|     AdaptableSnail      |     rear      |   660,348|    85,313|        69|      0.62|        0:01:35|
|         CRIM13          |     rear      |   179,132|    12,042|        69|      0.37|        0:02:03|
|         CRIM13          |   selfgroom   |   205,533|    14,472|        69|      0.36|        0:02:35|
|      CalMS21_task1      | genitalgroom  |   102,445|     6,270|        69|      0.66|        0:02:50|
|       ElegantMink       |     rear      |         0|         0|         0|         -|        0:02:51|
|       ElegantMink       |   selfgroom   |         0|         0|         0|         -|        0:02:52|
|       GroovyShrew       |     rear      |   899,280|    50,768|        69|      0.53|        0:04:39|
|       GroovyShrew       |     rest      |   530,886|    87,573|        69|      0.68|        0:05:30|
|       GroovyShrew       |   selfgroom   |   877,773|    22,893

In [ ]:
## (Pair) Huấn luyện và Kiểm định cho từng Lab-Behavior

groups = train_pair_behavior_dataframe.group_by("lab_id", "behavior", maintain_order=True)
total_groups = len(list(groups))
start_time = time.perf_counter()

for idx, ((lab_id, behavior), group) in tqdm(enumerate(groups), total=total_groups):
    if idx == 0:
        tqdm.write(
            f"|{'LAB':^25}|{'BEHAVIOR':^15}|{'SAMPLES':^10}|{'POSITIVE':^10}|{'FEATURES':^10}|{'F1':^10}|{'ELAPSED TIME':^15}|",
            end="\n",
        )

    tqdm.write(f"|{lab_id:^25}|{behavior:^15}|", end="")
    index_list = []
    feature_list = []
    label_list = []

    for row in group.rows(named=True):
        video_id = row["video_id"]
        agent = row["agent"]
        target = row["target"]

        agent_mouse_id = int(re.search(r"mouse(\d+)", agent).group(1))
        target_mouse_id = int(re.search(r"mouse(\d+)", target).group(1))

        data = pl.scan_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet").filter(
            (pl.col("agent_mouse_id") == agent_mouse_id) & (pl.col("target_mouse_id") == target_mouse_id)
        )
        index = data.select(INDEX_COLS).collect(engine="streaming")
        feature = data.select(pl.exclude(INDEX_COLS)).collect(engine="streaming")

        # Đọc dữ liệu nhãn (annotation)
        annotation_path = TRAIN_ANNOTATION_DIR / lab_id / f"{video_id}.parquet"
        if annotation_path.exists():
            annotation = (
                pl.scan_parquet(annotation_path)
                .filter(
                    (pl.col("action") == behavior)
                    & (pl.col("agent_id") == agent_mouse_id)
                    & (pl.col("target_id") == target_mouse_id)
                )
                .collect()
            )
        else:
            annotation = pl.DataFrame(
                schema={
                    "agent_id": pl.Int8,
                    "target_id": pl.Int8,
                    "action": str,
                    "start_frame": pl.Int16,
                    "stop_frame": pl.Int16,
                }
            )

        label_frames = set()
        for annotation_row in annotation.rows(named=True):
            label_frames.update(range(annotation_row["start_frame"], annotation_row["stop_frame"]))
        label = index.select(pl.col("video_frame").is_in(label_frames).cast(pl.Int8).alias("label"))

        if label.get_column("label").sum() == 0:
            continue

        index_list.append(index)
        feature_list.append(feature)
        label_list.append(label.get_column("label"))

    if not index_list:
        elapsed_time = datetime.timedelta(seconds=int(time.perf_counter() - start_time))
        tqdm.write(f"{0:>10,}|{0:>10,}|{0:>10,}|{'-':>10}|{str(elapsed_time):>15}|", end="\n")
        continue

    indices = pl.concat(index_list, how="vertical")
    features = pl.concat(feature_list, how="vertical")
    labels = pl.concat(label_list, how="vertical")

    del index_list, feature_list, label_list
    gc.collect()

    tqdm.write(f"{len(indices):>10,}|{labels.sum():>10,}|{len(features.columns):>10,}|", end="")

    f1 = train_validate(lab_id, behavior, indices, features, labels)
    tqdm.write(f"{f1:>10.2f}|", end="")

    elapsed_time = datetime.timedelta(seconds=int(time.perf_counter() - start_time))
    tqdm.write(f"{str(elapsed_time):>15}|", end="\n")

    gc.collect()

  0%|          | 0/104 [00:00<?, ?it/s]

|           LAB           |   BEHAVIOR    | SAMPLES  | POSITIVE | FEATURES |    F1    | ELAPSED TIME  |
|     AdaptableSnail      |   approach    | 1,524,536|     8,083|       149|      0.34|        0:05:01|
|     AdaptableSnail      |    attack     | 2,436,568|    28,759|       149|      0.18|        0:11:52|
|     AdaptableSnail      |     avoid     | 5,538,087|    22,923|       149|      0.17|        0:29:12|
|     AdaptableSnail      |     chase     | 3,707,542|    14,739|       149|      0.13|        0:39:58|
|     AdaptableSnail      |  chaseattack  | 1,217,344|     4,157|       149|      0.27|        0:43:31|
|     AdaptableSnail      |    submit     |   424,181|     8,478|       149|      0.39|        0:45:06|
|    BoisterousParrot     |   shepherd    | 9,504,414|    29,451|       149|      0.48|        1:11:53|
|         CRIM13          |   approach    |   205,533|    10,178|       149|      0.49|        1:12:47|
|         CRIM13          |    attack     |    71,906|     7,594

In [13]:
%%writefile robustify.py

def robustify(submission: pl.DataFrame, dataset: pl.DataFrame, train_test: str = "train"):
    traintest_directory = INPUT_DIR / f"{train_test}_tracking"

    old_submission = submission.clone()
    submission = submission.filter(pl.col("start_frame") < pl.col("stop_frame"))
    if len(submission) != len(old_submission):
        print("ERROR: Dropped frames with start >= stop")

    old_submission = submission.clone()
    group_list = []
    for _, group in submission.group_by("video_id", "agent_id", "target_id"):
        group = group.sort("start_frame")
        mask = np.ones(len(group), dtype=bool)
        last_stop_frame = 0
        for i, row in enumerate(group.rows(named=True)):
            if row["start_frame"] < last_stop_frame:
                mask[i] = False
            else:
                last_stop_frame = row["stop_frame"]
        group_list.append(group.filter(pl.Series("mask", mask)))

    submission = pl.concat(group_list)

    if len(submission) != len(old_submission):
        print("ERROR: Dropped duplicate frames")

    s_list = []
    for row in dataset.rows(named=True):
        lab_id = row["lab_id"]
        video_id = row["video_id"]
        if row["behaviors_labeled"] is None:
            continue

        if video_id in submission.get_column("video_id").to_list():
            continue

        if isinstance(row["behaviors_labeled"], str):
            continue

        print(f"Video {video_id} has no predictions.")

        path = traintest_directory / f"/{lab_id}/{video_id}.parquet"
        vid = pd.read_parquet(path)

        vid_behaviors = json.loads(row["behaviors_labeled"])
        vid_behaviors = sorted(list({b.replace("'", "") for b in vid_behaviors}))
        vid_behaviors = [b.split(",") for b in vid_behaviors]
        vid_behaviors = pd.DataFrame(vid_behaviors, columns=["agent", "target", "action"])

        start_frame = vid.video_frame.min()
        stop_frame = vid.video_frame.max() + 1

        for (agent, target), actions in vid_behaviors.groupby(["agent", "target"]):
            batch_length = int(np.ceil((stop_frame - start_frame) / len(actions)))
            for i, action_row in enumerate(actions.itertuples(index=False)):
                batch_start = start_frame + i * batch_length
                batch_stop = min(batch_start + batch_length, stop_frame)
                s_list.append((video_id, agent, target, action_row["action"], batch_start, batch_stop))

    if len(s_list) > 0:
        submission = pd.concat(
            [
                submission,
                pd.DataFrame(s_list, columns=["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"]),
            ]
        )
        print("ERROR: Filled empty videos")

    return submission

Writing robustify.py


In [ ]:
# Danh sách để lưu trữ kết quả dự đoán Out-of-Fold cho từng nhóm
group_oof_predictions = []

# Nhóm dữ liệu theo lab_id, video_id, agent, target
# maintain_order=True để giữ nguyên thứ tự ban đầu
groups = train_behavior_dataframe.group_by("lab_id", "video_id", "agent", "target", maintain_order=True)

# Thực hiện xử lý cho từng nhóm (hiển thị thanh tiến trình)
for (lab_id, video_id, agent, target), group in tqdm(groups, total=len(list(groups))):
    # Trích xuất ID chuột từ agent (chủ thể hành động)
    # Ví dụ: "mouse1" -> 1
    agent_mouse_id = int(re.search(r"mouse(\d+)", agent).group(1))
    
    # Trích xuất ID chuột từ target (đối tượng hành động)
    # Nếu là "self" (tự thân) thì là -1, ngược lại lấy ID chuột
    target_mouse_id = -1 if target == "self" else int(re.search(r"mouse(\d+)", target).group(1))

    # Danh sách lưu kết quả dự đoán cho từng loại hành vi trong nhóm này
    prediction_dataframe_list = []

    # Xử lý từng dòng (từng loại hành vi) trong nhóm
    for row in group.rows(named=True):
        behavior = row["behavior"]  # Loại hành vi (ví dụ: "grooming", "sniffing", v.v.)

        # Xây dựng đường dẫn file kết quả dự đoán OOF cho hành vi này
        oof_path = WORKING_DIR / "results" / lab_id / behavior / "oof_predictions.parquet"
        
        # Bỏ qua nếu file không tồn tại
        if not oof_path.exists():
            continue

        # Đọc kết quả dự đoán và lọc theo video_id, agent, target tương ứng
        prediction = (
            pl.scan_parquet(oof_path)  # Đọc trễ (Lazy loading - tiết kiệm bộ nhớ)
            .filter(
                (pl.col("video_id") == video_id)  # ID video khớp
                & (pl.col("agent_mouse_id") == agent_mouse_id)  # Chủ thể hành động khớp
                & (pl.col("target_mouse_id") == target_mouse_id)  # Đối tượng hành động khớp
            )
            .select(
                *INDEX_COLS,  # Chọn các cột index
                # Nhân xác suất dự đoán với nhãn dự đoán để tính điểm (score) cho hành vi này
                # Nếu nhãn dự đoán là 0 thì điểm cũng sẽ bằng 0
                (pl.col("prediction") * pl.col("predicted_label")).alias(behavior)
            )
            .collect()  # Thực thi việc đọc dữ liệu thực tế
        )

        # Bỏ qua nếu không có dòng nào sau khi lọc (không có dữ liệu tương ứng)
        if len(prediction) == 0:
            continue

        # Thêm kết quả dự đoán của hành vi này vào danh sách
        prediction_dataframe_list.append(prediction)

    # Bỏ qua nhóm này nếu không có kết quả dự đoán nào
    if not prediction_dataframe_list:
        continue

    # Nối kết quả dự đoán của nhiều hành vi theo chiều ngang
    # how="align": căn chỉnh dựa trên các cột index rồi nối lại
    prediction_dataframe = pl.concat(prediction_dataframe_list, how="align")

    # Lấy tên các cột ngoại trừ cột index (tức là tên các hành vi)
    cols = prediction_dataframe.select(pl.exclude(INDEX_COLS)).columns
    
    # Chọn hành vi có độ tin cậy cao nhất tại mỗi frame
    prediction_labels_dataframe = prediction_dataframe.with_columns(
        pl.struct(pl.exclude(INDEX_COLS))  # Gom điểm số của tất cả hành vi vào một struct
        .map_elements(
            # Hàm thực hiện cho mỗi dòng
            lambda row: "none" if sum(row.values()) == 0  # Nếu tổng điểm bằng 0 thì là "none"
                        else (cols[np.argmax(list(row.values()))]),  # Chọn hành vi có điểm cao nhất
            return_dtype=pl.String,
        )
        .alias("prediction")  # Đặt tên cột mới là "prediction"
    ).select(INDEX_COLS + ["prediction"])  # Chỉ giữ lại cột index và cột dự đoán

    # Gộp các hành vi giống nhau liên tiếp để xác định frame bắt đầu và kết thúc
    group_oof_prediction = (
        prediction_labels_dataframe
        .filter((pl.col("prediction") != pl.col("prediction").shift(1)))  # Chỉ lấy các dòng khác với dòng trước đó (điểm ranh giới)
        .with_columns(pl.col("video_frame").shift(-1).alias("stop_frame"))  # Lấy điểm ranh giới tiếp theo làm frame kết thúc
        .filter(pl.col("prediction") != "none")  # Loại bỏ trường hợp "none" (không có hành vi)
        .select(
            pl.col("video_id"),  # ID Video
            ("mouse" + pl.col("agent_mouse_id").cast(str)).alias("agent_id"),  # Chuyển đổi sang format "mouse1"
            # Nếu target_mouse_id là -1 thì là "self", ngược lại chuyển sang format "mouse2"
            pl.when(pl.col("target_mouse_id") == -1)
            .then(pl.lit("self"))
            .otherwise("mouse" + pl.col("target_mouse_id").cast(str))
            .alias("target_id"),
            pl.col("prediction").alias("action"),  # Tên hành vi
            pl.col("video_frame").alias("start_frame"),  # Frame bắt đầu
            pl.col("stop_frame"),  # Frame kết thúc
        )
    )

    # Thêm kết quả dự đoán đã xử lý của nhóm này vào danh sách tổng
    group_oof_predictions.append(group_oof_prediction)

# Chạy script "robustify.py" (xử lý hậu kỳ làm mượt kết quả)
%run -i robustify.py

oof_predictions = pl.concat(group_oof_predictions, how="vertical")
oof_predictions = robustify(oof_predictions, train_dataframe, train_test="train")
oof_predictions.with_row_index("row_id").write_csv(WORKING_DIR / "oof_predictions.csv")

  0%|          | 0/1442 [00:00<?, ?it/s]

ERROR: Dropped frames with start >= stop


<Figure size 640x480 with 0 Axes>

## Tính điểm (score) dựa trên dữ liệu kiểm tra

In [ ]:
# Tính toán chỉ số đánh giá (metric)
def compute_validation_metrics(submission, verbose=True):
    """Tính toán và hiển thị các chỉ số đánh giá cho hành vi đơn lẻ (single) và hành vi cặp (pair)."""
    # solution_df (Dữ liệu đáp án/đối chiếu)
    dataset = pl.read_csv(INPUT_DIR / "train.csv").to_pandas()

    solution = []
    for _, row in dataset.iterrows():
        lab_id = row["lab_id"]
        # Bỏ qua dữ liệu MABe22 nếu cần
        if lab_id.startswith("MABe22"):
            continue

        video_id = row["video_id"]
        path = TRAIN_ANNOTATION_DIR / lab_id / f"{video_id}.parquet"
        try:
            annot = pd.read_parquet(path)
        except FileNotFoundError:
            continue

        annot["lab_id"] = lab_id
        annot["video_id"] = video_id
        annot["behaviors_labeled"] = row["behaviors_labeled"]
        # Xử lý target_id: nếu khác agent_id thì thêm prefix "mouse", nếu trùng thì là "self"
        annot["target_id"] = np.where(
            annot.target_id != annot.agent_id, annot["target_id"].apply(lambda s: f"mouse{s}"), "self"
        )
        annot["agent_id"] = annot["agent_id"].apply(lambda s: f"mouse{s}")
        solution.append(annot)

    solution = pd.concat(solution)

    try:
        # Tách riêng hành vi đơn lẻ (single) và hành vi cặp (pair) từ file nộp
        submission_single = submission[submission["target_id"] == "self"].copy()
        submission_pair = submission[submission["target_id"] != "self"].copy()

        # Lọc dữ liệu đối chiếu (solution) để khớp với các video có trong file nộp (submission)
        solution_videos = set(submission["video_id"].unique())
        solution = solution[solution["video_id"].isin(solution_videos)]

        if len(solution) == 0:
            return

        # Tính điểm F1 tổng thể
        overall_f1 = score(solution, submission, "row_id", beta=1.0)
        print(f"\n{'=' * 60}")
        print("CHỈ SỐ HIỆU SUẤT (PERFORMANCE METRICS)")
        print(f"{'=' * 60}")
        print(f"Điểm F1 tổng thể: {overall_f1:.4f}")
        print(f"Tổng số dự đoán: {len(submission)}")
        print(f"  - Hành vi đơn lẻ (Single): {len(submission_single)}")
        print(f"  - Hành vi cặp (Pair): {len(submission_pair)}")

        # Tính điểm F1 cho từng hành động sử dụng hàm tính điểm hiện có
        solution_pl = pl.DataFrame(solution)
        submission_pl = pl.DataFrame(submission)

        # Thêm cột label_key và prediction_key để định danh duy nhất cho mỗi chuỗi hành động
        solution_pl = solution_pl.with_columns(
            pl.concat_str(
                [
                    pl.col("video_id").cast(pl.Utf8),
                    pl.col("agent_id").cast(pl.Utf8),
                    pl.col("target_id").cast(pl.Utf8),
                    pl.col("action"),
                ],
                separator="_",
            ).alias("label_key"),
        )
        submission_pl = submission_pl.with_columns(
            pl.concat_str(
                [
                    pl.col("video_id").cast(pl.Utf8),
                    pl.col("agent_id").cast(pl.Utf8),
                    pl.col("target_id").cast(pl.Utf8),
                    pl.col("action"),
                ],
                separator="_",
            ).alias("prediction_key"),
        )

        # Nhóm theo hành động và tính toán chỉ số
        action_stats = defaultdict(lambda: {"single": {"count": 0, "f1": 0.0}, "pair": {"count": 0, "f1": 0.0}})

        for lab in solution_pl["lab_id"].unique():
            lab_solution = solution_pl.filter(pl.col("lab_id") == lab).clone()
            lab_videos = set(lab_solution["video_id"].unique())
            lab_submission = submission_pl.filter(pl.col("video_id").is_in(lab_videos)).clone()

            # Tính F1 cho từng hành động dùng logic giống hàm single_lab_f1
            label_frames = defaultdict(set)
            prediction_frames = defaultdict(set)

            for row in lab_solution.to_dicts():
                label_frames[row["label_key"]].update(range(row["start_frame"], row["stop_frame"]))

            for row in lab_submission.to_dicts():
                key = row["prediction_key"]
                prediction_frames[key].update(range(row["start_frame"], row["stop_frame"]))

            for key in set(list(label_frames.keys()) + list(prediction_frames.keys())):
                action = key.split("_")[-1]
                mode = "single" if "self" in key else "pair"

                pred_frames = prediction_frames.get(key, set())
                label_frames_set = label_frames.get(key, set())

                tp = len(pred_frames & label_frames_set)
                fn = len(label_frames_set - pred_frames)
                fp = len(pred_frames - label_frames_set)

                if tp + fn + fp > 0:
                    f1 = (1 + 1**2) * tp / ((1 + 1**2) * tp + 1**2 * fn + fp)
                    action_stats[action][mode]["count"] += 1
                    action_stats[action][mode]["f1"] += f1

        # In tóm tắt hiệu suất theo từng hành động
        print("\nTóm tắt hiệu suất theo từng hành động:")
        print(f"{'-' * 60}")
        print(f"{'Hành động':<20} {'Chế độ':<10} {'Số lượng':<10} {'F1 TB':<10}")
        print(f"{'-' * 60}")

        for action in sorted(action_stats.keys()):
            for mode in ["single", "pair"]:
                stats = action_stats[action][mode]
                if stats["count"] > 0:
                    avg_f1 = stats["f1"] / stats["count"]
                    print(f"{action:<20} {mode:<10} {stats['count']:<10} {avg_f1:<10.4f}")

        # Tóm tắt theo chế độ (single/pair)
        single_actions = [a for a in action_stats.keys() if action_stats[a]["single"]["count"] > 0]
        pair_actions = [a for a in action_stats.keys() if action_stats[a]["pair"]["count"] > 0]

        if single_actions:
            single_avg_f1 = np.mean(
                [
                    action_stats[a]["single"]["f1"] / action_stats[a]["single"]["count"]
                    for a in single_actions
                    if action_stats[a]["single"]["count"] > 0
                ]
            )
            print(f"\nHành vi đơn lẻ (Single): {len(single_actions)} hành động, F1 TB: {single_avg_f1:.4f}")

        if pair_actions:
            pair_avg_f1 = np.mean(
                [
                    action_stats[a]["pair"]["f1"] / action_stats[a]["pair"]["count"]
                    for a in pair_actions
                    if action_stats[a]["pair"]["count"] > 0
                ]
            )
            print(f"Hành vi cặp (Pair): {len(pair_actions)} hành động, F1 TB: {pair_avg_f1:.4f}")

        print(f"{'=' * 60}\n")

    except Exception as e:
        if verbose:
            error_msg = str(e)
            if len(error_msg) > 200:
                error_msg = error_msg[:200] + "..."
            print(f"\nCảnh báo: Không thể tính toán chỉ số kiểm định: {error_msg}")
            if verbose:
                print(f"Traceback: {traceback.format_exc()[:300]}")

compute_validation_metrics(submission=pd.read_csv(WORKING_DIR / "oof_predictions.csv"))


PERFORMANCE METRICS
Overall F1 Score: 0.5036
Total predictions: 457826
  - Single behaviors: 114986
  - Pair behaviors: 342840

Per-Action Performance Summary:
------------------------------------------------------------
Action               Mode       Count      Avg F1    
------------------------------------------------------------
allogroom            pair       17         0.1635    
approach             pair       258        0.3672    
attack               pair       389        0.5586    
attemptmount         pair       42         0.1244    
avoid                pair       136        0.1487    
biteobject           single     16         0.0245    
chase                pair       117        0.1414    
chaseattack          pair       22         0.1741    
climb                single     30         0.3860    
defend               pair       64         0.3951    
dig                  single     60         0.3342    
disengage            pair       20         0.4260    
dominance      

In [16]:
import shutil, os

for root, dirs, files in os.walk("/kaggle/working/self_features"):
    for f in files:
        print("Deleting:", f)
        os.remove(os.path.join(root, f))

for root, dirs, files in os.walk("/kaggle/working/pair_features"):
    for f in files:
        print("Deleting:", f)
        os.remove(os.path.join(root, f))



Deleting: 379641005.parquet
Deleting: 918178571.parquet
Deleting: 278643799.parquet
Deleting: 1015291592.parquet
Deleting: 859139568.parquet
Deleting: 455201807.parquet
Deleting: 577720259.parquet
Deleting: 383642292.parquet
Deleting: 1096442159.parquet
Deleting: 5119542.parquet
Deleting: 1201849558.parquet
Deleting: 1142036064.parquet
Deleting: 2030735934.parquet
Deleting: 1937966765.parquet
Deleting: 304455631.parquet
Deleting: 1266015456.parquet
Deleting: 2134094751.parquet
Deleting: 1881725588.parquet
Deleting: 768075914.parquet
Deleting: 826874848.parquet
Deleting: 200061326.parquet
Deleting: 298840331.parquet
Deleting: 1149399074.parquet
Deleting: 793202924.parquet
Deleting: 2068280177.parquet
Deleting: 1108562754.parquet
Deleting: 1729143180.parquet
Deleting: 1013428301.parquet
Deleting: 921231021.parquet
Deleting: 1784646165.parquet
Deleting: 531131153.parquet
Deleting: 459610814.parquet
Deleting: 129168695.parquet
Deleting: 1549344783.parquet
Deleting: 1057262087.parquet
Delet

In [ ]:
# đọc dữ liệu test
test_dataframe = pl.read_csv(INPUT_DIR / "test.csv")

In [ ]:
# preprocess behavior labels
test_behavior_dataframe = (
    test_dataframe.filter(pl.col("behaviors_labeled").is_not_null())
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled").map_elements(eval, return_dtype=pl.List(pl.Utf8)).alias("behaviors_labeled_list"),
    )
    .explode("behaviors_labeled_list")
    .rename({"behaviors_labeled_list": "behaviors_labeled_element"})
    .select(
        pl.col("lab_id"),
        pl.col("video_id"),
        pl.col("behaviors_labeled_element").str.split(",").list[0].str.replace_all("'", "").alias("agent"),
        pl.col("behaviors_labeled_element").str.split(",").list[1].str.replace_all("'", "").alias("target"),
        pl.col("behaviors_labeled_element").str.split(",").list[2].str.replace_all("'", "").alias("behavior"),
    )
)

test_self_behavior_dataframe = test_behavior_dataframe.filter(pl.col("behavior").is_in(SELF_BEHAVIORS))
test_pair_behavior_dataframe = test_behavior_dataframe.filter(pl.col("behavior").is_in(PAIR_BEHAVIORS))

In [19]:
(WORKING_DIR / "self_features").mkdir(exist_ok=True, parents=True)
(WORKING_DIR / "pair_features").mkdir(exist_ok=True, parents=True)

rows = test_dataframe.rows(named=True)

for row in tqdm(rows, total=len(rows)):
    lab_id = row["lab_id"]
    video_id = row["video_id"]

    tracking_path = TEST_TRACKING_DIR / f"{lab_id}/{video_id}.parquet"
    tracking = pl.read_parquet(tracking_path)

    self_features = make_self_features(metadata=row, tracking=tracking)
    pair_features = make_pair_features(metadata=row, tracking=tracking)

    self_features.write_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet")
    pair_features.write_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet")

    del self_features, pair_features
    gc.collect()

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Danh sách lưu trữ kết quả dự đoán cho từng nhóm (tổ hợp lab_id, video_id, agent, target)
group_submissions = []

# Nhóm dữ liệu kiểm tra (test data) theo lab_id, video_id, agent, target
# maintain_order=True để giữ nguyên thứ tự ban đầu
groups = list(test_behavior_dataframe.group_by("lab_id", "video_id", "agent", "target", maintain_order=True))

# Thực hiện xử lý lần lượt cho từng nhóm (hiển thị thanh tiến trình)
for (lab_id, video_id, agent, target), group in tqdm(groups, total=len(list(groups))):
    # Trích xuất ID của agent (chuột thực hiện hành vi)
    # Ví dụ: "mouse1" -> 1
    agent_mouse_id = int(re.search(r"mouse(\d+)", agent).group(1))
    
    # Trích xuất ID của target (đối tượng của hành vi)
    # Nếu là "self" (hành vi tự thân) thì là -1, ngược lại lấy ID chuột
    # Ví dụ: "mouse2" -> 2, "self" -> -1
    target_mouse_id = -1 if target == "self" else int(re.search(r"mouse(\d+)", target).group(1))

    # ===== Đọc dữ liệu đặc trưng (Features) =====
    if target == "self":
        # Trường hợp hành vi tự thân (như rear, v.v.): đọc từ thư mục self_features
        
        # Đọc các cột index (video_id, agent_mouse_id, video_frame, v.v.)
        index = (
            pl.scan_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet")
            .filter((pl.col("agent_mouse_id") == agent_mouse_id))  # Lọc theo chuột agent
            .select(INDEX_COLS)  # Chỉ chọn các cột index
            .collect()  # Thực thi (lazy evaluation) để lấy dữ liệu
        )
        
        # Đọc các cột đặc trưng (tốc độ, khoảng cách, góc, v.v.)
        feature = (
            pl.scan_parquet(WORKING_DIR / "self_features" / f"{video_id}.parquet")
            .filter((pl.col("agent_mouse_id") == agent_mouse_id))
            .select(pl.exclude(INDEX_COLS))  # Chọn tất cả ngoại trừ cột index
            .collect()
        )
    else:
        # Trường hợp hành vi cặp (như attack, chase, v.v.): đọc từ thư mục pair_features
        
        # Đọc các cột index (lọc theo cả agent và target)
        index = (
            pl.scan_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet")
            .filter((pl.col("agent_mouse_id") == agent_mouse_id) & (pl.col("target_mouse_id") == target_mouse_id))
            .select(INDEX_COLS)
            .collect()
        )
        
        # Đọc các cột đặc trưng
        feature = (
            pl.scan_parquet(WORKING_DIR / "pair_features" / f"{video_id}.parquet")
            .filter((pl.col("agent_mouse_id") == agent_mouse_id) & (pl.col("target_mouse_id") == target_mouse_id))
            .select(pl.exclude(INDEX_COLS))
            .collect()
        )

    # Tạo DataFrame để chứa kết quả dự đoán (clone từ cột index)
    prediction_dataframe = index.clone()

    # ===== Thực hiện dự đoán cho từng hành vi (behavior) =====
    for row in group.rows(named=True):
        behavior = row["behavior"]  # Tên hành vi hiện tại (Ví dụ: "attack", "rear")

        # Danh sách lưu kết quả dự đoán của từng fold (phân chia kiểm chứng chéo)
        predictions = []  # Xác suất dự đoán
        prediction_labels = []  # Nhãn dự đoán (đã chuyển thành 0/1 bằng ngưỡng)

        # Lấy các thư mục fold đã lưu kết quả huấn luyện
        # Ví dụ: results/AdaptableSnail/attack/fold_0, fold_1, fold_2
        fold_dirs = list((WORKING_DIR / "results" / lab_id / behavior).glob("fold_*"))
        if not fold_dirs:
            # Bỏ qua nếu không tìm thấy mô hình đã huấn luyện
            continue

        # Thực hiện dự đoán với mô hình của từng fold
        for fold_dir in fold_dirs:
            # Đọc ngưỡng tối ưu đã lưu
            with open(fold_dir / "threshold.txt", "r") as f:
                threshold = float(f.read().strip())
            
            # Đọc mô hình XGBoost
            model = xgb.Booster(model_file=fold_dir / "model.json")
            
            # Chuyển đổi đặc trưng sang định dạng đầu vào của XGBoost (DMatrix)
            dtest = xgb.DMatrix(feature, feature_names=feature.columns)
            
            # Thực hiện dự đoán bằng mô hình (lấy giá trị xác suất)
            fold_predictions = model.predict(dtest)
            
            # Lưu xác suất dự đoán
            predictions.append(fold_predictions)
            
            # Áp dụng ngưỡng để gán nhãn (1: có hành vi, 0: không có hành vi)
            prediction_labels.append((fold_predictions >= threshold).astype(np.int8))

        # Thêm kết quả dự đoán vào DataFrame
        # Thêm cột là tích của "Xác suất dự đoán x Nhãn dự đoán" của từng fold
        # (Nếu nhãn là 0 thì xác suất về 0, nếu là 1 thì giữ nguyên xác suất)
        prediction_dataframe = prediction_dataframe.with_columns(
            *[
                pl.Series(name=f"{behavior}_{fold}", values=predictions[fold] * prediction_labels[fold], dtype=pl.Float32)
                for fold in range(len(fold_dirs))
            ]
        )

    # ===== Chọn hành vi có xác suất cao nhất =====
    
    # Lấy tên các cột ngoại trừ cột index (tức là các cột dự đoán hành vi)
    cols = prediction_dataframe.select(pl.exclude(INDEX_COLS)).columns
    if not cols:
        # Nếu không có cột dự đoán nào thì cảnh báo và bỏ qua
        tqdm.write(f"Warning: No predictions found for {lab_id}, {video_id}, {agent}, {target}")
        continue

    # Chọn hành vi có xác suất cao nhất tại mỗi frame
    prediction_labels_dataframe = prediction_dataframe.with_columns(
        pl.struct(pl.col(cols))  # Gom tất cả cột dự đoán vào struct
        .map_elements(
            # Hàm xử lý cho mỗi dòng (frame):
            # - Nếu tất cả giá trị dự đoán bằng 0 -> "none" (không có hành vi)
            # - Ngược lại, trả về tên hành vi có giá trị lớn nhất (lấy phần tên trước dấu gạch dưới)
            lambda row: "none" if sum(row.values()) == 0 else (cols[np.argmax(list(row.values()))]).split("_")[0],
            return_dtype=pl.String,
        )
        .alias("prediction")  # Đặt tên cột mới là "prediction"
    ).select(INDEX_COLS + ["prediction"])  # Chỉ giữ lại cột index và cột dự đoán

    # ===== Gom các hành vi giống nhau liên tiếp thành sự kiện (Events) =====
    
    group_submission = (
        prediction_labels_dataframe
        .filter((pl.col("prediction") != pl.col("prediction").shift(1)))  # Chỉ giữ lại frame mà hành vi có sự thay đổi
        .with_columns(pl.col("video_frame").shift(-1).alias("stop_frame"))  # Lấy điểm thay đổi tiếp theo làm frame kết thúc
        .filter(pl.col("prediction") != "none")  # Loại bỏ "none" (không có hành vi)
        .select(
            # Chọn và chuyển đổi cột cho phù hợp với định dạng nộp bài
            pl.col("video_id"),
            ("mouse" + pl.col("agent_mouse_id").cast(str)).alias("agent_id"),  # Ví dụ: 1 -> "mouse1"
            pl.when(pl.col("target_mouse_id") == -1)  # Nếu target_mouse_id là -1
            .then(pl.lit("self"))  # Chuyển thành "self"
            .otherwise("mouse" + pl.col("target_mouse_id").cast(str))  # Ngược lại là "mouseN"
            .alias("target_id"),
            pl.col("prediction").alias("action"),  # Tên hành vi
            pl.col("video_frame").alias("start_frame"),  # Frame bắt đầu
            pl.col("stop_frame"),  # Frame kết thúc
        )
    )

    # Thêm dữ liệu nộp của nhóm này vào danh sách
    group_submissions.append(group_submission)

# ===== Kết hợp kết quả dự đoán của tất cả các nhóm =====

# Nối dữ liệu nộp của tất cả các nhóm theo chiều dọc và sắp xếp
submission = pl.concat(group_submissions, how="vertical").sort(
    "video_id",
    "agent_id",
    "target_id",
    "action",
    "start_frame",
    "stop_frame",
)

# Xử lý làm sạch/kiện toàn dữ liệu nộp (xóa trùng lặp, sửa lỗi frame, v.v.)
submission = robustify(submission, test_dataframe, train_test="test")

# Thêm số thứ tự dòng (row_id) và lưu thành file CSV
submission.with_row_index("row_id").write_csv(WORKING_DIR / "submission.csv")

  0%|          | 0/16 [00:00<?, ?it/s]

ERROR: Dropped frames with start >= stop


In [21]:
!head submission.csv

row_id,video_id,agent_id,target_id,action,start_frame,stop_frame
0,438887472,mouse4,self,rear,161,182
1,438887472,mouse4,self,rear,183,184
2,438887472,mouse4,self,rear,265,266
3,438887472,mouse4,self,rear,270,272
4,438887472,mouse4,self,rear,275,292
5,438887472,mouse4,self,rear,328,329
6,438887472,mouse4,self,rear,332,335
7,438887472,mouse4,self,rear,338,342
8,438887472,mouse4,self,rear,343,357
